# 제품 비교 분석 종합 인사이트 도출


## 목표
- `analysis_outputs/` 폴더의 비교 분석 텍스트 + 시각화 이미지를 종합 분석
- Gemini API를 활용한 멀티모달 분석 (텍스트 + 이미지)
- 구조화된 JSON 형식의 실행 가능한 비즈니스 인사이트 도출

## 분석 구성
1. 제품별 종합 순위 및 평가
2. 측면별 상세 분석 (품질, 디자인, 가성비 등)
3. 감정 분석 인사이트
4. 주요 트렌드 및 패턴
5. 실행 가능한 개선 제안
6. 시장 분석 (포지셔닝, 경쟁 우위, 위험 요소)

 ## 1. 환경 설정 및 라이브러리 임포트

In [ ]:
# 필요한 라이브러리 설치
!pip install google-genai python-dotenv pydantic

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel, Field, ValidationError
from typing import List, Optional
import time

# 환경 변수 로드
load_dotenv()

# API 설정
api_key = os.getenv('GEMINI_API_KEY')
gemini_model = 'gemini-2.5-pro'

# API 키 검증
if not api_key or 'YOUR_API_KEY' in api_key:
    raise ValueError("경고: .env 파일에서 GEMINI_API_KEY를 실제 API 키로 설정해주세요!")

print("API 키 설정 완료")
print(f"사용 모델: {gemini_model}")

# Gemini 클라이언트 초기화
client = genai.Client(api_key=api_key)

# 폴더 경로 설정
ANALYSIS_DIR = Path("analysis_outputs")

print(f"\n분석 폴더: {ANALYSIS_DIR}")
print(f"폴더 존재: {ANALYSIS_DIR.exists()}")


## 2. 인사이트 응답 스키마 정의

Gemini API가 반환할 JSON 구조를 Pydantic으로 정의합니다.

In [ ]:
# =============================================================================
# 2-1. 제품 순위 스키마
# =============================================================================

class ProductRanking(BaseModel):
    """제품별 종합 순위 및 평가"""
    rank: int = Field(description="종합 순위 (1위부터)")
    product_name: str = Field(description="제품명")
    overall_score: float = Field(
        ge=0, le=100, 
        description="종합 점수 (0-100점)"
    )
    strengths: List[str] = Field(
        description="주요 강점 (3-5개)",
        min_length=3,
        max_length=5
    )
    weaknesses: List[str] = Field(
        description="주요 약점 (3-5개)",
        min_length=3,
        max_length=5
    )
    recommended_for: str = Field(
        description="추천 대상 (어떤 사용자에게 적합한지, 100자 이내)",
        max_length=100
    )

print("ProductRanking 스키마 정의 완료")

In [ ]:
# =============================================================================
# 2-2. 측면별 인사이트 스키마
# =============================================================================

class AspectInsight(BaseModel):
    """측면별 상세 인사이트"""
    aspect: str = Field(description="분석 측면 (품질, 사용성, 가성비, 기능성, 디자인, 내구성, 고객서비스, 배송, 포장, 전반적)")
    best_product: str = Field(description="해당 측면에서 가장 우수한 제품")
    worst_product: str = Field(description="해당 측면에서 가장 부족한 제품")
    key_finding: str = Field(
        description="해당 측면에 대한 핵심 발견사항 (150자 이내)",
        max_length=150
    )

print("AspectInsight 스키마 정의 완료")

In [ ]:
# =============================================================================
# 2-3. 감정 인사이트 스키마
# =============================================================================

class SentimentInsight(BaseModel):
    """감정 분석 인사이트"""
    most_positive_product: str = Field(description="가장 긍정적인 반응을 받은 제품")
    most_negative_product: str = Field(description="가장 부정적인 반응을 받은 제품")
    common_complaints: List[str] = Field(
        description="공통 불만사항 (3-5개)",
        min_length=3,
        max_length=5
    )
    common_praises: List[str] = Field(
        description="공통 칭찬사항 (3-5개)",
        min_length=3,
        max_length=5
    )

print("SentimentInsight 스키마 정의 완료")

In [ ]:
# =============================================================================
# 2-4. 트렌드 및 제안 스키마
# =============================================================================

class KeyTrend(BaseModel):
    """주요 트렌드 및 패턴"""
    trend_name: str = Field(description="트렌드 이름")
    description: str = Field(
        description="트렌드 설명 (150자 이내)",
        max_length=150
    )
    affected_products: List[str] = Field(description="영향받는 제품들")


class ActionableRecommendation(BaseModel):
    """실행 가능한 제안"""
    target_product: str = Field(description="대상 제품")
    category: str = Field(description="개선 카테고리 (품질/디자인/서비스/마케팅 등)")
    recommendation: str = Field(
        description="구체적인 개선 제안 (200자 이내)",
        max_length=200
    )
    expected_impact: str = Field(
        description="예상 효과 (100자 이내)",
        max_length=100
    )
    priority: str = Field(description="우선순위 (high/medium/low)")

print("KeyTrend 및 ActionableRecommendation 스키마 정의 완료")

In [ ]:
# =============================================================================
# 2-5. 종합 인사이트 스키마
# =============================================================================

class ComprehensiveInsight(BaseModel):
    """종합 인사이트 분석 결과"""
    
    executive_summary: str = Field(
        description="전체 제품 비교 분석 요약 (500자 이내)",
        max_length=500
    )
    
    product_rankings: List[ProductRanking] = Field(
        description="제품별 종합 순위 및 평가",
        min_length=1
    )
    
    aspect_insights: List[AspectInsight] = Field(
        description="측면별 상세 인사이트 (품질, 사용성, 가성비, 기능성, 전반적은 필수. 디자인, 내구성, 고객서비스, 배송, 포장은 데이터 있으면 포함)",
        min_length=5,  # 최소 5개 (핵심 측면들)
        max_length=10  # 최대 10개 (모든 측면)
    )
    
    sentiment_insights: SentimentInsight = Field(
        description="감정 분석 인사이트"
    )
    
    key_trends: List[KeyTrend] = Field(
        description="주요 트렌드 및 패턴 (3-5개)",
        min_length=3,
        max_length=5
    )
    
    actionable_recommendations: List[ActionableRecommendation] = Field(
        description="실행 가능한 개선 제안 (5-10개)",
        min_length=5,
        max_length=10
    )
    
    market_positioning: str = Field(
        description="시장 포지셔닝 분석 (각 제품의 시장 내 위치, 500자 이내)",
        max_length=500
    )
    
    competitive_advantages: str = Field(
        description="경쟁 우위 요소 분석 (500자 이내)",
        max_length=500
    )
    
    risk_factors: List[str] = Field(
        description="주의해야 할 위험 요소 (3-5개)",
        min_length=3,
        max_length=5
    )

print("ComprehensiveInsight 종합 스키마 정의 완료")

 ## 3. 시스템 프롬프트 정의

In [ ]:
INSIGHT_SYSTEM_INSTRUCTION = """
당신은 제품 비교 분석 전문가입니다.
제품 리뷰 데이터와 시각화 자료를 분석하여 실행 가능한 비즈니스 인사이트를 도출하세요.

[분석 원칙]

1. 데이터 기반 판단
   - 제공된 텍스트 요약과 시각화 이미지를 모두 활용
   - 구체적인 수치와 근거 제시
   - 추측보다는 데이터에서 확인된 사실 중심

2. 실용적 인사이트
   - 비즈니스 의사결정에 즉시 활용 가능
   - 구체적이고 실행 가능한 제안
   - 우선순위를 명확히 구분

3. 균형잡힌 시각
   - 장점과 단점을 공정하게 평가
   - 각 제품의 고유한 가치 인정
   - 타겟 고객별 차별화된 추천

4. 트렌드 파악
   - 데이터에서 나타나는 패턴 식별
   - 시장 방향성 예측
   - 경쟁 우위 요소 분석

[분석 항목별 가이드]

executive_summary (경영진 요약)
- 전체 분석의 핵심을 2-3문장으로 압축
- 가장 중요한 발견사항과 권장사항 포함

product_rankings (제품 순위)
- overall_score: 평점, 추천도, 리뷰 수, 감정 분포를 종합 평가
- strengths: 데이터에서 확인된 구체적 강점
- weaknesses: 개선이 필요한 명확한 약점
- recommended_for: 구체적인 타겟 고객 명시

aspect_insights (측면별 인사이트) - 매우 중요!
**반드시 다음 측면들을 모두 분석해야 합니다:**

필수 분석 측면 (최소 5개):
1. 품질 (quality) - 제품의 전반적인 품질, 성분, 안전성
2. 사용성 (usability) - 사용 편의성, 사용감, 자극 여부
3. 가성비 (price_value) - 가격 대비 만족도
4. 기능성 (functionality) - 제품의 성능, 효과
5. 전반적 (overall) - 종합적인 만족도

추가 분석 측면 (데이터에 있으면 포함):
6. 디자인 (design) - 외관, 용기 디자인
7. 내구성 (durability) - 지속성, 보관 안정성
8. 고객서비스 (customer_service) - A/S, 응대
9. 배송 (shipping) - 배송 속도, 포장 상태
10. 포장 (packaging) - 용기, 패키징 품질

각 측면마다:
- best_product: 데이터상 가장 높은 만족도를 보인 제품
- worst_product: 데이터상 가장 낮은 만족도를 보인 제품
- key_finding: 해당 측면의 시장 트렌드나 중요 발견사항

주의: aspect_insights는 최소 5개 이상 작성해야 합니다!

sentiment_insights (감정 인사이트)
- 긍정/부정 반응이 가장 극명한 제품 식별
- common_complaints: 여러 제품에 공통적인 불만
- common_praises: 여러 제품에 공통적인 칭찬

key_trends (주요 트렌드)
- 데이터에서 발견되는 패턴
- 예: "높은 가격대 제품일수록 내구성 평가가 높음"

actionable_recommendations (실행 가능한 제안)
- 각 제품별로 구체적 개선 방향
- priority: 개선의 시급성과 영향도 고려
  * high: 즉시 개선 필요, 큰 영향
  * medium: 중요하지만 시급하지 않음
  * low: 장기적 개선 과제

market_positioning (시장 포지셔닝)
- 각 제품이 시장에서 차지하는 위치
- 예: "프리미엄 시장 리더", "가성비 최강자"

competitive_advantages (경쟁 우위)
- 각 제품만의 차별화된 강점
- 경쟁자 대비 우위 요소

risk_factors (위험 요소)
- 주의해야 할 부정적 신호
- 예: "지속적인 품질 불만 증가", "신규 경쟁자 등장"

[작성 가이드]
- 명확하고 간결한 표현
- 전문 용어보다는 이해하기 쉬운 언어
- 수치와 근거를 명시
- "~인 것 같다" 대신 "~로 나타났다" 사용
- 텍스트 요약 데이터의 "측면별 만족도" 섹션을 반드시 참고하여 각 측면을 분석
"""

print("시스템 프롬프트 정의 완료")

 ## 4. 데이터 로드 함수

In [ ]:
def load_summary_text(file_path: Path) -> str:
    """
    제품 비교 요약 텍스트 파일 로드
    
    Args:
        file_path: 텍스트 파일 경로
        
    Returns:
        str: 파일 내용
    """
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        print(f"텍스트 파일 로드 완료: {file_path.name}")
        print(f"  길이: {len(content)} 문자")
        return content
    except Exception as e:
        print(f"텍스트 파일 로드 실패: {e}")
        return ""

print("load_summary_text 함수 정의 완료")

In [ ]:
def load_images_from_directory(directory: Path) -> List[types.Part]:
    """
    디렉토리에서 모든 이미지를 로드하여 Gemini API 형식으로 변환
    
    Args:
        directory: 이미지가 있는 디렉토리 경로
        
    Returns:
        List[types.Part]: Gemini API용 이미지 파트 리스트
    """
    image_parts = []
    
    if not directory.exists():
        print(f"경고: {directory} 디렉토리를 찾을 수 없습니다.")
        return image_parts
    
    # 지원하는 이미지 확장자
    image_extensions = {'.png', '.jpg', '.jpeg', '.webp'}
    
    # 이미지 파일 찾기
    image_files = sorted([
        f for f in directory.iterdir() 
        if f.suffix.lower() in image_extensions
    ])
    
    print(f"\n{len(image_files)}개의 시각화 이미지 발견:")
    
    for image_path in image_files:
        try:
            # 이미지를 바이트로 읽기
            with open(image_path, 'rb') as f:
                image_bytes = f.read()
            
            # MIME 타입 결정
            mime_type = {
                '.jpg': 'image/jpeg',
                '.jpeg': 'image/jpeg',
                '.png': 'image/png',
                '.webp': 'image/webp'
            }.get(image_path.suffix.lower(), 'image/jpeg')
            
            # Gemini API 파트 생성
            image_part = types.Part.from_bytes(
                data=image_bytes,
                mime_type=mime_type
            )
            
            image_parts.append(image_part)
            print(f"  - {image_path.name}")
            
        except Exception as e:
            print(f"  오류: {image_path.name} - {e}")
    
    return image_parts

print("load_images_from_directory 함수 정의 완료")

 ## 5. 인사이트 생성 함수

In [ ]:
def generate_comprehensive_insight(
    summary_text: str,
    image_parts: List[types.Part],
    max_retries: int = 2
) -> ComprehensiveInsight:
    """
    제품 비교 요약 텍스트와 시각화 이미지를 분석하여 종합 인사이트 생성
    (오류 메시지 피드백 기반 재시도)
    
    Args:
        summary_text: 제품 비교 요약 텍스트
        image_parts: 시각화 이미지 파트 리스트
        max_retries: 최대 재시도 횟수 (기본 2회)
        
    Returns:
        ComprehensiveInsight: 종합 인사이트 분석 결과
    """
    
    print("\n" + "=" * 80)
    print("종합 인사이트 생성 시작")
    print("=" * 80)
    
    # 기본 프롬프트
    base_prompt = f"""
다음은 여러 제품에 대한 리뷰 비교 분석 결과입니다.

## 텍스트 요약 데이터:
{summary_text}

## 시각화 차트:
위에 첨부된 {len(image_parts)}개의 시각화 이미지를 참고하세요.
각 차트는 다음 정보를 포함합니다:
- 제품별 평균 평점 및 추천도
- 리뷰 수 비교
- 감정 분포
- 측면별 긍정/부정 비율
- 키워드 분석 (장점/단점 워드클라우드)

## 요청사항:
이 데이터를 종합적으로 분석하여 다음을 도출하세요:

1. 제품 순위: 종합 점수를 계산하여 순위 매기기
2. 측면별 분석: 품질, 디자인, 가성비 등 각 측면별 최고/최악 제품
3. 감정 인사이트: 긍정/부정 반응 패턴
4. 트렌드 파악: 데이터에서 발견되는 주요 패턴
5. 실행 제안: 각 제품별 구체적 개선 방안
6. 시장 분석: 포지셔닝, 경쟁 우위, 위험 요소

반드시 첨부된 데이터에 기반하고 텍스트 데이터와 시각화 차트를 모두 활용하여 심층 분석해주세요.
"""
    
    # Generation Config
    generation_config = types.GenerateContentConfig(
        temperature=0.3,
        response_mime_type='application/json',
        response_schema=ComprehensiveInsight,
        system_instruction=INSIGHT_SYSTEM_INSTRUCTION
    )
    
    # 오류 메시지 저장
    last_error_message = None
    
    # 재시도 루프
    for attempt in range(max_retries + 1):
        try:
            print(f"\n[시도 {attempt + 1}/{max_retries + 1}] API 호출 중...")
            
            # 프롬프트 구성 (오류 메시지 포함)
            if last_error_message:
                prompt = base_prompt + f"""

**재시도 {attempt}회 - 이전 응답에서 오류 발생**:
{last_error_message}

위 오류를 수정하여 스키마에 맞게 다시 응답해주세요!
특히 다음 사항을 확인하세요:
- 모든 필수 필드가 포함되어 있는지
- 리스트 필드의 최소/최대 길이가 지켜지는지
- 숫자 필드의 범위가 올바른지 (예: overall_score는 0-100)
- priority 필드는 'high', 'medium', 'low' 중 하나인지
"""
            else:
                prompt = base_prompt
            
            # 콘텐츠 구성: 이미지들 + 프롬프트
            contents = image_parts + [prompt]
            
            # API 호출
            response = client.models.generate_content(
                model=gemini_model,
                contents=contents,
                config=generation_config
            )
            
            # 결과 파싱
            print("API 호출 성공")
            print("결과 파싱 중...")
            
            # parsed 속성 확인
            if response.parsed:
                print("파싱 성공!")
                return response.parsed
            else:
                # 텍스트에서 JSON 추출 시도
                response_text = response.text
                
                # JSON 파싱
                insight_data = json.loads(response_text)
                
                # Pydantic 모델로 변환
                insight = ComprehensiveInsight(**insight_data)
                print("파싱 성공!")
                return insight
        
        except ValidationError as e:
            # Pydantic 검증 오류
            last_error_message = f"데이터 검증 오류:\n{str(e)}"
            
            if attempt < max_retries:
                print(f"  경고: 검증 오류 발생, 재시도 중...")
                print(f"  오류 내용: {str(e)[:200]}...")
                time.sleep(4)
                continue
            else:
                print(f"  오류: 최대 재시도 횟수 초과")
                print(f"  마지막 오류: {str(e)}")
                raise
        
        except json.JSONDecodeError as e:
            # JSON 파싱 오류
            last_error_message = f"JSON 파싱 오류:\n{str(e)}\n응답이 올바른 JSON 형식이 아닙니다."
            
            if attempt < max_retries:
                print(f"  경고: JSON 파싱 오류, 재시도 중...")
                print(f"  오류 위치: {str(e)[:200]}...")
                time.sleep(4)
                continue
            else:
                print(f"  오류: JSON 파싱 실패")
                print(f"  마지막 오류: {str(e)}")
                raise
        
        except Exception as e:
            # 기타 오류
            last_error_message = f"예상치 못한 오류:\n{str(e)}"
            
            if attempt < max_retries:
                print(f"  경고: 오류 발생, 재시도 중...")
                print(f"  오류 내용: {str(e)[:200]}...")
                time.sleep(4)
                continue
            else:
                print(f"  오류: 분석 실패")
                print(f"  마지막 오류: {str(e)}")
                raise

print("generate_comprehensive_insight 함수 정의 완료 (오류 처리 및 재시도 로직 포함)")

 ## 6. 결과 저장 및 출력 함수

In [ ]:
def save_and_display_insight(
    insight: ComprehensiveInsight,
    output_dir: Path
):
    """
    인사이트 결과를 저장하고 출력
    
    Args:
        insight: 생성된 인사이트 객체
        output_dir: 결과를 저장할 디렉토리
    """
    
    output_dir.mkdir(exist_ok=True)
    
    # 1. JSON 저장
    json_path = output_dir / "comprehensive_insight.json"
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(insight.model_dump(), f, ensure_ascii=False, indent=2)
    print(f"\nJSON 저장: {json_path}")
    
    # 2. 마크다운 리포트 생성
    md_path = output_dir / "comprehensive_insight_report.md"
    
    md_content = f"""# 제품 비교 분석 종합 인사이트 리포트

---

## 경영진 요약

{insight.executive_summary}

---

## 제품 순위 및 평가

"""
    
    # 제품 순위
    for ranking in insight.product_rankings:
        md_content += f"""
### {ranking.rank}위: {ranking.product_name}
**종합 점수**: {ranking.overall_score}/100점

**주요 강점:**
"""
        for strength in ranking.strengths:
            md_content += f"- {strength}\n"
        
        md_content += "\n**주요 약점:**\n"
        for weakness in ranking.weaknesses:
            md_content += f"- {weakness}\n"
        
        md_content += f"\n**추천 대상**: {ranking.recommended_for}\n\n---\n"
    
    # 측면별 인사이트
    md_content += "\n## 측면별 상세 인사이트\n\n"
    for aspect in insight.aspect_insights:
        md_content += f"""
### {aspect.aspect}
- **최고 제품**: {aspect.best_product}
- **최악 제품**: {aspect.worst_product}
- **핵심 발견**: {aspect.key_finding}

"""
    
    # 감정 인사이트
    md_content += f"""
---

## 감정 분석 인사이트

**가장 긍정적인 제품**: {insight.sentiment_insights.most_positive_product}

**가장 부정적인 제품**: {insight.sentiment_insights.most_negative_product}

**공통 칭찬사항:**
"""
    for praise in insight.sentiment_insights.common_praises:
        md_content += f"- {praise}\n"
    
    md_content += "\n**공통 불만사항:**\n"
    for complaint in insight.sentiment_insights.common_complaints:
        md_content += f"- {complaint}\n"
    
    # 주요 트렌드
    md_content += "\n---\n\n## 주요 트렌드 및 패턴\n\n"
    for trend in insight.key_trends:
        md_content += f"""
### {trend.trend_name}
{trend.description}

**영향받는 제품**: {', '.join(trend.affected_products)}

"""
    
    # 실행 가능한 제안
    md_content += "\n---\n\n## 실행 가능한 개선 제안\n\n"
    
    # 우선순위별로 정렬
    high_priority = [r for r in insight.actionable_recommendations if r.priority == 'high']
    medium_priority = [r for r in insight.actionable_recommendations if r.priority == 'medium']
    low_priority = [r for r in insight.actionable_recommendations if r.priority == 'low']
    
    if high_priority:
        md_content += "### 높은 우선순위\n\n"
        for rec in high_priority:
            md_content += f"""
**[{rec.target_product}] {rec.category}**
- 제안: {rec.recommendation}
- 예상 효과: {rec.expected_impact}

"""
    
    if medium_priority:
        md_content += "### 중간 우선순위\n\n"
        for rec in medium_priority:
            md_content += f"""
**[{rec.target_product}] {rec.category}**
- 제안: {rec.recommendation}
- 예상 효과: {rec.expected_impact}

"""
    
    if low_priority:
        md_content += "### 낮은 우선순위\n\n"
        for rec in low_priority:
            md_content += f"""
**[{rec.target_product}] {rec.category}**
- 제안: {rec.recommendation}
- 예상 효과: {rec.expected_impact}

"""
    
    # 시장 분석
    md_content += f"""
---

## 시장 분석

### 시장 포지셔닝
{insight.market_positioning}

### 경쟁 우위 요소
{insight.competitive_advantages}

### 주의해야 할 위험 요소
"""
    for risk in insight.risk_factors:
        md_content += f"- {risk}\n"
    
    md_content += "\n---\n\n*본 리포트는 AI 기반 자동 분석 결과입니다.*\n"
    
    # 마크다운 저장
    with open(md_path, 'w', encoding='utf-8') as f:
        f.write(md_content)
    print(f"마크다운 리포트 저장: {md_path}")
    
    # 3. 콘솔 출력
    print("\n" + "=" * 80)
    print("종합 인사이트 분석 결과")
    print("=" * 80)
    print(md_content)

print("save_and_display_insight 함수 정의 완료")

 ## 7. 메인 실행

In [ ]:
# =============================================================================
# 데이터 로드
# =============================================================================

print("\n" + "=" * 80)
print("Step 1: 데이터 로드")
print("=" * 80)

# 텍스트 파일 로드
summary_file = ANALYSIS_DIR / "product_comparison_summary.txt"
summary_text = load_summary_text(summary_file)

# 이미지 파일 로드
image_parts = load_images_from_directory(ANALYSIS_DIR)

print(f"\n로드 완료:")
print(f"  - 텍스트: {len(summary_text)} 문자")
print(f"  - 이미지: {len(image_parts)} 개")

In [ ]:
# =============================================================================
# 인사이트 생성
# =============================================================================

print("\n" + "=" * 80)
print("Step 2: 인사이트 생성")
print("=" * 80)

insight = generate_comprehensive_insight(
    summary_text=summary_text,
    image_parts=image_parts
)

print("\n인사이트 생성 완료!")

In [ ]:
# =============================================================================
# 결과 저장 및 출력
# =============================================================================
print("\n" + "=" * 80)
print("Step 3: 결과 저장 및 출력")
print("=" * 80)

save_and_display_insight(
    insight=insight,
    output_dir=ANALYSIS_DIR
)

print("\n" + "=" * 80)
print("전체 프로세스 완료!")
print("=" * 80)

## 8. 결과 확인


생성된 파일:
- `analysis_outputs/comprehensive_insight.json` - JSON 형식 원본 데이터
- `analysis_outputs/comprehensive_insight_report.md` - 마크다운 리포트

주요 인사이트:
- 제품별 순위 및 종합 점수
- 측면별 최고/최악 제품
- 감정 분석 (긍정/부정 패턴)
- 주요 트렌드
- 실행 가능한 개선 제안 (우선순위별)
- 시장 분석 및 위험 요소